In [1]:
import numpy as np
import zarr

from tqdm  import tqdm

In [2]:
from zarro import (
    DataKind,
    WriteType,
    ControlVar,
    AxisSpecs,
    ReadoutSpecs,
    MeasurementSchema,
    ZarrWriter,
    NpzWriter
)

## **Examples of 2D scans**

In [3]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))
Vgt = ControlVar(name="Vgt", values=np.linspace(1, 2, 10, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpecs([Vgt]),
        AxisSpecs([Vch, Vm])
    ],
    measure = [
        ReadoutSpecs(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpecs(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpecs(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24"

### **2D scan / save all at once in the end of scan**

use `writer.write_all(dict)`

In [4]:
mschema.writetype = WriteType.ALL

writer = ZarrWriter(
    root=root_path,
    experiment_id="0001_vna_scan",
    schema=mschema,
    overwrite=True,
)

In [ ]:
MAG_All = []
LOCKIN_All = []
TEMP_All = []

for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    LOCKIN_All.append(LOCKIN)
    TEMP_All.append(TEMP)
    MAG_All.append(MAG)

writer.write_all({"S21": MAG_All, "temperature": TEMP_All, "lockin_X": LOCKIN_All})
writer.add_array("Freqs", freq, meta={"unit": "Hz", "description": "Frequency points for S21 measurement"})

100%|██████████| 10/10 [00:00<00:00, 250.68it/s]


In [6]:
writer.tags = ["vna", "temperature", "lockin", "chipid_1234"]
writer.write_metadata()

### **2D scan / save trace at each step**

use `writer.write_trace((i,), dict)`

In [8]:
mschema.writetype = WriteType.TRACEWISE

writer = ZarrWriter(
    root=root_path,
    experiment_id="0001_vna_scan",
    schema=mschema,
    overwrite=True,
)

In [9]:
for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    writer.write_trace((i,), {"S21": MAG, "temperature": TEMP, "lockin_X": LOCKIN})

100%|██████████| 10/10 [00:00<00:00, 104.04it/s]


### **2D scan / save trace at each point**

use `writer.write_point((i,j), dict)`

if the writing is slow, try to adjust `writer.target_file_size_bytes` in a way, that `shard_shape` is as close to `chunk_shape`

In [11]:
mschema.writetype = WriteType.POINTWISE

writer = ZarrWriter(
    root=root_path,
    experiment_id="0001_vna_scan",
    schema=mschema,
    target_file_size_bytes=1*1024*1024,  # 5 MB
    overwrite=True,
)
print(writer._dataset_map["S21"].info)

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200, 1601)
Shard shape        : (1, 166, 1601)
Chunk shape        : (1, 1, 1601)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 12808000 (12.2M)


In [12]:
for i in tqdm(range(len(Vgt.values))):
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()
        writer.write_point((i, j), {"S21": mag, "temperature": T1K, "lockin_X": lockinx})

100%|██████████| 10/10 [00:07<00:00,  1.39it/s]


## **Examples of 1D scans**

In [13]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpecs([Vch, Vm])
    ],
    measure = [
        ReadoutSpecs(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpecs(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpecs(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24"

### **1D scan / save all at once in the end of scan**

use `writer.write_all(dict)`

In [14]:
mschema.writetype = WriteType.ALL

writer = ZarrWriter(
    root=root_path,
    experiment_id="0001_vna_scan",
    schema=mschema,
    overwrite=True,
)

In [15]:
MAG = []
LOCKIN = []
TEMP = []
for j in range(len(Vch.values)):
    freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
    lockinx = np.random.random()
    T1K = np.random.random()

    MAG.append(mag)
    LOCKIN.append(lockinx)
    TEMP.append(T1K)

writer.write_all({"S21": MAG, "temperature": TEMP, "lockin_X": LOCKIN})

### **1D scan / save data at each point**

use `writer.write_point((i,), dict)`

if the writing is slow, try to adjust `writer.target_file_size_bytes` in a way, that `shard_shape` is as close to `chunk_shape`

In [17]:
mschema.writetype = WriteType.POINTWISE

writer = ZarrWriter(
    root=root_path,
    experiment_id="0001_vna_scan",
    schema=mschema,
    target_file_size_bytes=1*1024*1024,  # 5 MB
    overwrite=True,
)

In [18]:
for j in range(len(Vch.values)):
    freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
    lockinx = np.random.random()
    T1K = np.random.random()

    writer.write_point((j,), {"S21": mag, "temperature": T1K, "lockin_X": lockinx})

## **Writing npz_like**

In [25]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))
Vgt = ControlVar(name="Vgt", values=np.linspace(1, 2, 200, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpecs([Vgt]),
        AxisSpecs([Vch, Vm])
    ],
    measure = [
        ReadoutSpecs(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpecs(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpecs(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24"


writer = ZarrWriter(
    root=root_path,
    experiment_id="0002_vna_scan",
    schema=mschema,
    initialize_arrays=False,
    overwrite=True,
)

In [26]:
MAG_All = []
LOCKIN_All = []
TEMP_All = []

for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    LOCKIN_All.append(LOCKIN)
    TEMP_All.append(TEMP)
    MAG_All.append(MAG)

writer.write_npzlike({"S21": MAG_All, "temperature": TEMP_All, "lockin_X": LOCKIN_All})

writer.add_array("Freqs", freq, meta={"unit": "Hz", "description": "Frequency points for S21 measurement"})

100%|██████████| 200/200 [00:00<00:00, 445.24it/s]


In [27]:
writer.tags = ["vna", "temperature", "lockin", "chipid_1234"]
writer.write_metadata()

## **Reading data**

In [28]:
z2 = zarr.open(root_path + "/0002_vna_scan" + '/vault.zarr', mode='r')


In [30]:
list(z2.keys())

['lockin_X', 'Freqs', 'S21', 'temperature']

In [31]:
list(z2.attrs.keys())

['date', 'experiment_id', 'schema', 'tags']

In [32]:
z2["S21"][:,159, 8]

array([3.89782349e-01, 9.12650085e-01, 2.66776023e-01, 9.04123339e-02,
       5.16193004e-01, 8.58694232e-01, 4.15756965e-01, 8.57484203e-01,
       5.98002721e-01, 5.96066868e-01, 9.99193546e-01, 9.49493935e-01,
       4.11412064e-01, 7.30138882e-04, 2.36081886e-01, 1.49778563e-01,
       2.09803688e-01, 1.02261562e-02, 5.55274189e-01, 1.35491613e-01,
       4.30312869e-01, 8.15092811e-01, 3.23801441e-01, 3.99769657e-01,
       6.98144643e-01, 6.98307326e-01, 9.28664935e-01, 5.87990479e-01,
       4.90175356e-01, 7.04782844e-01, 8.35268198e-01, 9.75465803e-01,
       5.90064672e-01, 7.33608006e-01, 2.23986334e-01, 6.42924616e-01,
       1.26704983e-01, 1.66689219e-01, 4.12067579e-02, 3.86381350e-01,
       6.84272851e-01, 4.73843032e-01, 8.31797505e-01, 8.47906133e-01,
       7.57614438e-01, 1.39944174e-01, 2.33105473e-01, 7.12075667e-01,
       8.54259653e-03, 9.54040174e-01, 2.17137475e-01, 4.39321880e-01,
       1.19384924e-01, 3.83903189e-01, 9.65514513e-01, 9.89809213e-02,
      

## **Using NpzWriter**

In [37]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))
Vgt = ControlVar(name="Vgt", values=np.linspace(1, 2, 200, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpecs([Vgt]),
        AxisSpecs([Vch, Vm])
    ],
    measure = [
        ReadoutSpecs(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpecs(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpecs(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24/0003_vna_scan_npz"

writer = NpzWriter(
    root=root_path,
    schema=mschema,
    compressed=False
)

In [39]:
MAG_All = []
LOCKIN_All = []
TEMP_All = []

for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    LOCKIN_All.append(LOCKIN)
    TEMP_All.append(TEMP)
    MAG_All.append(MAG)

writer.write({"S21": MAG_All, "temperature": TEMP_All, "lockin_X": LOCKIN_All})

#writer.add_array("Freqs", freq, meta={"unit": "Hz", "description": "Frequency points for S21 measurement"})

100%|██████████| 200/200 [00:00<00:00, 472.77it/s]


In [42]:
z3 = np.load(root_path + "/data.npz")

In [45]:
z3["S21"][:, 145, 5]

array([0.39288064, 0.56297704, 0.3441343 , 0.06008765, 0.69573836,
       0.03970992, 0.23455588, 0.98549763, 0.72534655, 0.15558074,
       0.55095144, 0.52071333, 0.04315569, 0.46100566, 0.32134783,
       0.57458275, 0.93800945, 0.69900453, 0.95767242, 0.30330512,
       0.0990119 , 0.04282771, 0.10119846, 0.06956759, 0.31003121,
       0.00550859, 0.1600852 , 0.81081519, 0.39573676, 0.26912325,
       0.10325973, 0.15657941, 0.72357967, 0.09888905, 0.31219129,
       0.56546272, 0.10942856, 0.9501032 , 0.54086702, 0.30893644,
       0.72793208, 0.97950279, 0.75441762, 0.96418415, 0.94590832,
       0.42341196, 0.19548983, 0.97542742, 0.96337372, 0.38150552,
       0.20363111, 0.17670403, 0.24272405, 0.43304896, 0.7715785 ,
       0.54351048, 0.05634617, 0.91215329, 0.07876411, 0.82989168,
       0.12043685, 0.76017768, 0.57399347, 0.78193795, 0.44180916,
       0.62655613, 0.79339311, 0.10101776, 0.63463443, 0.451455  ,
       0.46111887, 0.85207248, 0.11202904, 0.23691969, 0.26083